In [3]:
import json
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import (
    context_precision,
    context_recall,
    faithfulness,
    answer_correctness
)

def run_ragas_eval(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    ds = Dataset.from_list(data)


    return evaluate(
        dataset=ds,
        metrics=[context_precision, context_recall, faithfulness, answer_correctness]
    )

# Run all evaluations
naive_scores_bm25 = run_ragas_eval("RAG_JSON/naive_rag_evaluation_set_bm25.json")
naive_scores_faiss = run_ragas_eval("RAG_JSON/naive_rag_evaluation_set_faiss.json")
advanced_scores = run_ragas_eval("RAG_JSON/advanced_rag_evaluation_set.json")

print("Naive RAG (BM25):", naive_scores_bm25)
print("Naive RAG (FAISS):", naive_scores_faiss)
print("Advanced RAG:", advanced_scores)


Evaluating: 100%|██████████| 32/32 [03:29<00:00,  6.56s/it]


Naive RAG (BM25): {'context_precision': 0.8646, 'context_recall': 0.6312, 'faithfulness': 0.9007, 'answer_correctness': 0.6032}
Naive RAG (FAISS): {'context_precision': 0.8505, 'context_recall': 0.7500, 'faithfulness': 0.7998, 'answer_correctness': 0.6252}
Advanced RAG: {'context_precision': 0.8507, 'context_recall': 0.8063, 'faithfulness': 0.7769, 'answer_correctness': 0.7620}


In [8]:
import os
import json
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import (
    context_precision,
    context_recall,
    faithfulness,
    answer_correctness
)

BASE_DIR = "RAG_JSON"
FUSION_DIR = os.path.join(BASE_DIR, "fusion_results")

def run_ragas_eval(file_path):
    """Run RAGAS evaluation on a JSON file and return results."""
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    ds = Dataset.from_list(data)

    return evaluate(
        dataset=ds,
        metrics=[context_precision, context_recall, faithfulness, answer_correctness]
    )

def evaluate_all():
    results = {}

    # === Evaluate naive files (BM25 + FAISS) ===
    naive_files = {
        "naive_bm25": os.path.join(BASE_DIR, "naive_rag_evaluation_set_bm25.json"),
        "naive_faiss": os.path.join(BASE_DIR, "naive_rag_evaluation_set_faiss.json"),
    }

    for name, path in naive_files.items():
        if os.path.exists(path):
            print(f"🔍 Evaluating {name} ...")
            results[name] = run_ragas_eval(path)
        else:
            print(f"⚠️ Skipped {name}, file not found.")

    # === Evaluate all fusion result folders (01–09) ===
    if os.path.exists(FUSION_DIR):
        for folder in sorted(os.listdir(FUSION_DIR)):
            folder_path = os.path.join(FUSION_DIR, folder)
            if not os.path.isdir(folder_path):
                continue

            # Detect file name automatically
            json_file = None
            for f in os.listdir(folder_path):
                if f.startswith("rag_evaluation_set_") and f.endswith(".json"):
                    json_file = os.path.join(folder_path, f)
                    break

            if json_file and os.path.exists(json_file):
                label = f"fusion_{folder}"
                print(f"🔍 Evaluating {label} ({json_file}) ...")
                results[label] = run_ragas_eval(json_file)
            else:
                print(f"⚠️ Skipped {folder_path}, rag_evaluation_set_XX.json not found.")
    else:
        print("⚠️ No fusion_results directory found.")

    # === Print summary ===
    print("\n📊 Evaluation Summary:")
    for name, score in results.items():
        # Try multiple ways to convert result
        if hasattr(score, "to_dict"):
            score = score.to_dict()
        elif isinstance(score, list) and hasattr(score[0], "to_dict"):
            score = {m.metric_name: m.to_dict() for m in score}
        elif not isinstance(score, dict):
            score = vars(score)

        print(f"\n{name}:")
        for k, v in score.items():
            try:
                print(f"  {k:<20}: {float(v):.4f}")
            except Exception:
                print(f"  {k:<20}: {v}")

    print("\n✅ All evaluations completed successfully.")


if __name__ == "__main__":
    evaluate_all()


🔍 Evaluating naive_bm25 ...


Evaluating: 100%|██████████| 32/32 [03:19<00:00,  6.25s/it]


🔍 Evaluating naive_faiss ...


Evaluating: 100%|██████████| 32/32 [03:03<00:00,  5.74s/it]


🔍 Evaluating fusion_01 (RAG_JSON\fusion_results\01\rag_evaluation_set_01.json) ...


...

🔍 Evaluating fusion_08 (RAG_JSON\fusion_results\08\rag_evaluation_set_08.json) ...


Evaluating: 100%|██████████| 32/32 [03:03<00:00,  5.73s/it]


🔍 Evaluating fusion_09 (RAG_JSON\fusion_results\09\rag_evaluation_set_09.json) ...


Evaluating: 100%|██████████| 32/32 [03:14<00:00,  6.08s/it]



📊 Evaluation Summary:

naive_bm25:
  scores              : [{'context_precision': 0.99999999998, 'context_recall': 0.75, 'faithfulness': 0.9545454545454546, 'answer_correctness': 0.5645419575066787}, {'context_precision': 0.9166666666361111, 'context_recall': 1.0, 'faithfulness': 1.0, 'answer_correctness': 0.5111013603777049}, {'context_precision': 0.99999999998, 'context_recall': 1.0, 'faithfulness': 0.75, 'answer_correctness': 0.4393559350358739}, {'context_precision': 0.99999999998, 'context_recall': 0.8, 'faithfulness': 0.9444444444444444, 'answer_correctness': 0.46711735173182667}, {'context_precision': 0.0, 'context_recall': 0.2, 'faithfulness': 1.0, 'answer_correctness': 0.7993056556876531}, {'context_precision': 0.99999999998, 'context_recall': 0.0, 'faithfulness': 0.8181818181818182, 'answer_correctness': 0.57044079612662}, {'context_precision': 0.9999999999, 'context_recall': 0.5, 'faithfulness': 0.9545454545454546, 'answer_correctness': 0.5017489421619982}, {'context_precis

In [9]:
import json
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import (
    context_precision,
    context_recall,
    faithfulness,
    answer_correctness
)


def run_ragas_eval(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    ds = Dataset.from_list(data)

    return evaluate(
        dataset=ds,
        metrics=[context_precision, context_recall, faithfulness, answer_correctness]
    )




# === Run all evaluations ===

naive_bm25 = run_ragas_eval("RAG_JSON/naive_rag_evaluation_set_bm25.json")
naive_faiss = run_ragas_eval("RAG_JSON/naive_rag_evaluation_set_faiss.json")

fusion_01 = run_ragas_eval("RAG_JSON/fusion_results/01/rag_evaluation_set_01.json")
fusion_02 = run_ragas_eval("RAG_JSON/fusion_results/02/rag_evaluation_set_02.json")
fusion_03 = run_ragas_eval("RAG_JSON/fusion_results/03/rag_evaluation_set_03.json")
fusion_04 = run_ragas_eval("RAG_JSON/fusion_results/04/rag_evaluation_set_04.json")
fusion_05 = run_ragas_eval("RAG_JSON/fusion_results/05/rag_evaluation_set_05.json")
fusion_06 = run_ragas_eval("RAG_JSON/fusion_results/06/rag_evaluation_set_06.json")
fusion_07 = run_ragas_eval("RAG_JSON/fusion_results/07/rag_evaluation_set_07.json")
fusion_08 = run_ragas_eval("RAG_JSON/fusion_results/08/rag_evaluation_set_08.json")
fusion_09 = run_ragas_eval("RAG_JSON/fusion_results/09/rag_evaluation_set_09.json")

# === Print all results ===

print("Naive RAG (BM25)", naive_bm25)
print("Naive RAG (FAISS)", naive_faiss)
print("Fusion 01", fusion_01)
print("Fusion 02", fusion_02)
print("Fusion 03", fusion_03)
print("Fusion 04", fusion_04)
print("Fusion 05", fusion_05)
print("Fusion 06", fusion_06)
print("Fusion 07", fusion_07)
print("Fusion 08", fusion_08)
print("Fusion 09", fusion_09)


Evaluating:  50%|█████     | 16/32 [03:04<03:22, 12.65s/it]Exception raised in Job[3]: TimeoutError()
Exception raised in Job[7]: TimeoutError()
Exception raised in Job[0]: TimeoutError()
Exception raised in Job[12]: TimeoutError()
Evaluating:  47%|████▋     | 15/32 [02:41<02:44,  9.67s/it]Exception raised in Job[7]: TimeoutError()
Exception raised in Job[15]: TimeoutError()
Exception raised in Job[3]: TimeoutError()
Exception raised in Job[11]: TimeoutError()
Evaluating:  72%|███████▏  | 23/32 [02:54<00:36,  4.06s/it]Exception raised in Job[3]: TimeoutError()
Exception raised in Job[15]: TimeoutError()
Evaluating: 100%|██████████| 32/32 [03:11<00:00,  5.97s/it]


Naive RAG (BM25) {'context_precision': 0.7778, 'context_recall': 0.6562, 'faithfulness': 0.8778, 'answer_correctness': 0.5946}
Naive RAG (FAISS) {'context_precision': 0.8687, 'context_recall': 0.7500, 'faithfulness': 0.7711, 'answer_correctness': 0.5476}
Fusion 01 {'context_precision': 0.8571, 'context_recall': 0.7750, 'faithfulness': 0.8730, 'answer_correctness': 0.9315}
Fusion 02 {'context_precision': 0.8507, 'context_recall': 0.7500, 'faithfulness': 0.9164, 'answer_correctness': 0.6895}
Fusion 03 {'context_precision': 0.8583, 'context_recall': 0.7750, 'faithfulness': 0.9055, 'answer_correctness': 0.7395}
Fusion 04 {'context_precision': 0.8750, 'context_recall': 0.7750, 'faithfulness': 0.8667, 'answer_correctness': 0.6837}
Fusion 05 {'context_precision': 0.8507, 'context_recall': 0.7750, 'faithfulness': 0.8350, 'answer_correctness': 0.7803}
Fusion 06 {'context_precision': 0.8750, 'context_recall': 0.7750, 'faithfulness': 0.7665, 'answer_correctness': 0.6551}
Fusion 07 {'context_preci

In [ ]:
import json
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import (
    context_precision,
    context_recall,
    faithfulness,
    answer_correctness
)


def run_ragas_eval(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    ds = Dataset.from_list(data)

    return evaluate(
        dataset=ds,
        metrics=[context_precision, context_recall, faithfulness, answer_correctness]
    )




# === Run all evaluations ===

naive_bm25 = run_ragas_eval("NEW_RAG_JSON/new_naive_rag_evaluation_set_bm25.json")
naive_faiss = run_ragas_eval("NEW_RAG_JSON/new_naive_rag_evaluation_set_faiss.json")

ablation_no_hybrid_bm25 = run_ragas_eval("NEW_RAG_JSON/new_rag_ablation_no_hybrid_bm25_rerank_stepback.json")
ablation_no_hybrid_faiss = run_ragas_eval("NEW_RAG_JSON/new_rag_ablation_no_hybrid_faiss_rerank_stepback.json")
ablation_no_stepack = run_ragas_eval("NEW_RAG_JSON/new_rag_ablation_no_stepback_hybrid_rerank.json")
ablation_no_rerank = run_ragas_eval("NEW_RAG_JSON/new_rag_ablation_no_rerank_hybrid_stepback.json")

fusion_01 = run_ragas_eval("NEW_RAG_JSON/new_fusion_results/01/rag_evaluation_set_01.json")
fusion_02 = run_ragas_eval("NEW_RAG_JSON/new_fusion_results/02/rag_evaluation_set_02.json")
fusion_03 = run_ragas_eval("NEW_RAG_JSON/new_fusion_results/03/rag_evaluation_set_03.json")
fusion_04 = run_ragas_eval("NEW_RAG_JSON/new_fusion_results/04/rag_evaluation_set_04.json")
fusion_05 = run_ragas_eval("NEW_RAG_JSON/new_fusion_results/05/rag_evaluation_set_05.json")
fusion_06 = run_ragas_eval("NEW_RAG_JSON/new_fusion_results/06/rag_evaluation_set_06.json")
fusion_07 = run_ragas_eval("NEW_RAG_JSON/new_fusion_results/07/rag_evaluation_set_07.json")
fusion_08 = run_ragas_eval("NEW_RAG_JSON/new_fusion_results/08/rag_evaluation_set_08.json")
fusion_09 = run_ragas_eval("NEW_RAG_JSON/new_fusion_results/09/rag_evaluation_set_09.json")

# === Print all results ===

print("Naive RAG (BM25)", naive_bm25)
print("Naive RAG (FAISS)", naive_faiss)
print("Ablation No Hybrid BM25", ablation_no_hybrid_bm25)
print("Ablation No Hybrid FAISS", ablation_no_hybrid_faiss)
print("Ablation No Stepback", ablation_no_stepack)
print("Ablation No Rerank", ablation_no_rerank)
print("Fusion 01", fusion_01)
print("Fusion 02", fusion_02)
print("Fusion 03", fusion_03)
print("Fusion 04", fusion_04)
print("Fusion 05", fusion_05)
print("Fusion 06", fusion_06)
print("Fusion 07", fusion_07)
print("Fusion 08", fusion_08)
print("Fusion 09", fusion_09)


Evaluating:  31%|███       | 27/88 [03:18<06:39,  6.55s/it]Exception raised in Job[19]: APIConnectionError(Connection error.)
Exception raised in Job[23]: APIConnectionError(Connection error.)
Evaluating: 100%|██████████| 88/88 [08:23<00:00,  5.72s/it]


Naive RAG (BM25) {'context_precision': 0.9384, 'context_recall': 0.6848, 'faithfulness': 0.7596, 'answer_correctness': 0.6436}
Naive RAG (FAISS) {'context_precision': 0.9977, 'context_recall': 0.6689, 'faithfulness': 0.7853, 'answer_correctness': 0.6628}
Ablation No Hybrid BM25 {'context_precision': 0.9860, 'context_recall': 0.6939, 'faithfulness': 0.8009, 'answer_correctness': 0.6593}
Ablation No Hybrid FAISS {'context_precision': 0.9889, 'context_recall': 0.6758, 'faithfulness': 0.7869, 'answer_correctness': 0.6613}
Ablation No Stepback {'context_precision': 0.9881, 'context_recall': 0.7144, 'faithfulness': 0.8306, 'answer_correctness': 0.6808}
Ablation No Rerank {'context_precision': 0.9841, 'context_recall': 0.6758, 'faithfulness': 0.7894, 'answer_correctness': 0.6423}
Fusion 01 {'context_precision': 0.9841, 'context_recall': 0.6848, 'faithfulness': 0.8169, 'answer_correctness': 0.6001}
Fusion 02 {'context_precision': 0.9864, 'context_recall': 0.6939, 'faithfulness': 0.8283, 'answe

In [ ]:
import json
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import (
    context_precision,
    context_recall,
    faithfulness,
    answer_correctness
)


def run_ragas_eval(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    ds = Dataset.from_list(data)

    return evaluate(
        dataset=ds,
        metrics=[context_precision, context_recall, faithfulness, answer_correctness]
    )

# === Run all evaluations ===

naive_bm25 = run_ragas_eval("NEW_EXPANDED_RAG_JSON/new_expanded_naive_rag_evaluation_set_bm25.json")
naive_faiss = run_ragas_eval("NEW_EXPANDED_RAG_JSON/new_expanded_naive_rag_evaluation_set_faiss.json")

ablation_no_hybrid_bm25 = run_ragas_eval("NEW_EXPANDED_RAG_JSON/new_expanded_rag_ablation_no_hybrid_bm25_rerank_stepback.json")
ablation_no_hybrid_faiss = run_ragas_eval("NEW_EXPANDED_RAG_JSON/new_expanded_rag_ablation_no_hybrid_faiss_rerank_stepback.json")
ablation_no_stepack = run_ragas_eval("NEW_EXPANDED_RAG_JSON/new_expanded_rag_ablation_no_stepback_hybrid_rerank.json")
ablation_no_rerank = run_ragas_eval("NEW_EXPANDED_RAG_JSON/new_expanded_rag_ablation_no_rerank_hybrid_stepback.json")

In [ ]:
fusion_01 = run_ragas_eval("NEW_EXPANDED_RAG_JSON/new_expanded_fusion_results/01/rag_evaluation_set_01.json")
fusion_02 = run_ragas_eval("NEW_EXPANDED_RAG_JSON/new_expanded_fusion_results/02/rag_evaluation_set_02.json")
fusion_03 = run_ragas_eval("NEW_EXPANDED_RAG_JSON/new_expanded_fusion_results/03/rag_evaluation_set_03.json")
fusion_04 = run_ragas_eval("NEW_EXPANDED_RAG_JSON/new_expanded_fusion_results/04/rag_evaluation_set_04.json")
fusion_05 = run_ragas_eval("NEW_EXPANDED_RAG_JSON/new_expanded_fusion_results/05/rag_evaluation_set_05.json")
fusion_06 = run_ragas_eval("NEW_EXPANDED_RAG_JSON/new_expanded_fusion_results/06/rag_evaluation_set_06.json")
fusion_07 = run_ragas_eval("NEW_EXPANDED_RAG_JSON/new_expanded_fusion_results/07/rag_evaluation_set_07.json")
fusion_08 = run_ragas_eval("NEW_EXPANDED_RAG_JSON/new_expanded_fusion_results/08/rag_evaluation_set_08.json")
fusion_09 = run_ragas_eval("NEW_EXPANDED_RAG_JSON/new_expanded_fusion_results/09/rag_evaluation_set_09.json")

# === Print all results ===

print("Naive RAG (BM25)", naive_bm25)
print("Naive RAG (FAISS)", naive_faiss)
print("Ablation No Hybrid BM25", ablation_no_hybrid_bm25)
print("Ablation No Hybrid FAISS", ablation_no_hybrid_faiss)
print("Ablation No Stepback", ablation_no_stepack)
print("Ablation No Rerank", ablation_no_rerank)
print("Fusion 01", fusion_01)
print("Fusion 02", fusion_02)
print("Fusion 03", fusion_03)
print("Fusion 04", fusion_04)
print("Fusion 05", fusion_05)
print("Fusion 06", fusion_06)
print("Fusion 07", fusion_07)
print("Fusion 08", fusion_08)
print("Fusion 09", fusion_09)

In [5]:
naive_bm25_df = naive_bm25.to_pandas()
naive_faiss_df = naive_faiss.to_pandas()
ablation_no_hybrid_bm25_df = ablation_no_hybrid_bm25.to_pandas()
ablation_no_hybrid_faiss_df = ablation_no_hybrid_faiss.to_pandas()
ablation_no_stepack_df = ablation_no_stepack.to_pandas()
ablation_no_rerank_df = ablation_no_rerank.to_pandas()
fusion_01_df = fusion_01.to_pandas()
fusion_02_df = fusion_02.to_pandas()
fusion_03_df = fusion_03.to_pandas()
fusion_04_df = fusion_04.to_pandas()
fusion_05_df = fusion_05.to_pandas()
fusion_06_df = fusion_06.to_pandas()
fusion_07_df = fusion_07.to_pandas()
fusion_09_df = fusion_09.to_pandas()
fusion_08_df = fusion_08.to_pandas()

In [6]:
naive_bm25_df.to_csv('evaluation_df/naive_bm25.csv')
naive_faiss_df.to_csv('evaluation_df/naive_faiss.csv')
ablation_no_hybrid_bm25_df.to_csv('evaluation_df/ablation_no_hybrid_bm25.csv')
ablation_no_hybrid_faiss_df.to_csv('evaluation_df/ablation_no_hybrid_faiss.csv')
ablation_no_stepack_df.to_csv('evaluation_df/ablation_no_stepack.csv')
ablation_no_rerank_df.to_csv('evaluation_df/ablation_no_rerank.csv')
fusion_01_df.to_csv('evaluation_df/fusion_01.csv')
fusion_02_df.to_csv('evaluation_df/fusion_02.csv')
fusion_03_df.to_csv('evaluation_df/fusion_03.csv')
fusion_04_df.to_csv('evaluation_df/fusion_04.csv')
fusion_05_df.to_csv('evaluation_df/fusion_05.csv')
fusion_06_df.to_csv('evaluation_df/fusion_06.csv')
fusion_07_df.to_csv('evaluation_df/fusion_07.csv')
fusion_09_df.to_csv('evaluation_df/fusion_09.csv')
fusion_08_df.to_csv('evaluation_df/fusion_08.csv')